# Phase 2.1: Random Forest Optimization for Improved Failure Detection Recall

**Project:** Machine Learning System for Early Detection of Equipment Failure  
**Dataset:** AI4I 2020 Predictive Maintenance  
**Primary Production Algorithm:** Optimized Random Forest with Domain Physics Telemetry  
**Target:** `Machine failure` (0 = Normal, 1 = Failure)  

---

### Key Objectives
1. **Preserve Random Forest as Primary Algorithm:** Maintain Random Forest as the final production model.
2. **Address Baseline Recall Limitation:** Overcome the baseline 59.3% recall while avoiding excessive false alarms.
3. **Evaluate 6 Optimization Strategies:**
   - Baseline class-weighted Random Forest
   - Exp A: Hyperparameter tuning (tree depth, min leaf/split, estimators)
   - Exp B: Operational decision threshold calibration
   - Exp C: SMOTE within ImbPipeline cross-validation folds
   - Exp D: Balanced Random Forest (bootstrap downsampling per tree)
   - Exp E: Physics-based domain telemetry derivations (Delta-T, Mechanical Power, Overstrain Index)
   - Exp F: Combined Physics Pipeline + Calibrated Threshold
4. **Held-Out Test Validation:** Evaluate once on 2,000 untouched test samples.
5. **Interpretability & Deployment Interface:** Verify feature importance and 5-sensor frontend compatibility.


In [ ]:
import os
import sys
import json
from pathlib import Path

sys.path.insert(0, str(Path("../src").resolve()))
from utils import PRIMARY_FEATURES, TARGET, ENGINEERED_FEATURE_NAMES, ALL_8_FEATURES, add_physics_features

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import FunctionTransformer
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, fbeta_score,
    roc_auc_score, average_precision_score, confusion_matrix, classification_report
)
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.ensemble import BalancedRandomForestClassifier
from imblearn.over_sampling import SMOTE

print('Environment and packages initialized successfully!')


## 1. Load Preprocessed Partitions & Existing Metrics

We load the exact splits prepared in Phase 1 without altering any data.

In [ ]:
splits_dir = Path("../data/splits")
X_train = pd.read_csv(splits_dir / "X_train.csv", index_col=0)
X_test = pd.read_csv(splits_dir / "X_test.csv", index_col=0)
y_train = pd.read_csv(splits_dir / "y_train.csv", index_col=0).squeeze()
y_test = pd.read_csv(splits_dir / "y_test.csv", index_col=0).squeeze()

print(f"X_train shape: {X_train.shape} | Failures: {y_train.sum()} ({y_train.mean()*100:.2f}%)")
print(f"X_test shape:  {X_test.shape}  | Failures: {y_test.sum()}  ({y_test.mean()*100:.2f}%)")
print(f"Features: {list(X_train.columns)}")


## 2. Experimental Optimization Results Comparison

We load the measured validation results across all 6 experimental configurations.

In [ ]:
metrics_dir = Path("../outputs/metrics")
exp_df = pd.read_csv(metrics_dir / "rf_experiments_comparison.csv")
display_cols = ["Config_Name", "CV_PR-AUC_Mean", "Val_Recall", "Val_Precision", "Val_F1-Score", "Val_F2-Score", "Val_PR-AUC", "Val_FP", "Val_FN"]
exp_df[display_cols]


## 3. Threshold Optimization Curves

Let's examine how the decision threshold influences Failure Detection Recall and Alarm Precision.

In [ ]:
thresh_df = pd.read_csv(metrics_dir / "rf_threshold_analysis.csv")
eng_thresh = thresh_df[thresh_df["Config"].str.contains("Engineered", case=False)]

plt.figure(figsize=(9, 5))
plt.plot(eng_thresh["Threshold"], eng_thresh["Recall"], 'r-o', label="Recall (Failure Catch Rate)")
plt.plot(eng_thresh["Threshold"], eng_thresh["Precision"], 'b-s', label="Precision (Alarm Accuracy)")
plt.plot(eng_thresh["Threshold"], eng_thresh["F2-Score"], 'g-D', label="F2-Score (Cost Metric)")
plt.axvline(0.35, color="green", linestyle="--", label="Selected Threshold (0.35)")
plt.title("Decision Threshold Sweep for Physics-Engineered Random Forest")
plt.xlabel("Probability Threshold")
plt.ylabel("Score")
plt.grid(True)
plt.legend()
plt.show()


## 4. Final Held-Out Test Evaluation (2,000 Records)

We evaluate the serialized winning production pipeline on the untouched test partition.

In [ ]:
best_model_path = Path("../models/best_model/ai4i_failure_model.joblib")
production_model = joblib.load(best_model_path)

test_proba = production_model.predict_proba(X_test)[:, 1]
test_pred = (test_proba >= 0.35).astype(int)

tn, fp, fn, tp = confusion_matrix(y_test, test_pred).ravel()
print(f"Test Recall:    {tp / (tp + fn) * 100:.2f}% ({tp}/{tp + fn} caught)")
print(f"Test Precision: {tp / (tp + fp) * 100:.2f}% (Only {fp} false alarms)")
print(f"Test F1-Score:  {f1_score(y_test, test_pred):.4f}")
print(f"Test F2-Score:  {fbeta_score(y_test, test_pred, beta=2):.4f}")
print(f"Test PR-AUC:    {average_precision_score(y_test, test_proba):.4f}")
print(f"Test ROC-AUC:   {roc_auc_score(y_test, test_proba):.4f}")

plt.figure(figsize=(6, 5))
sns.heatmap([[tn, fp], [fn, tp]], annot=True, fmt="d", cmap="Greens",
            xticklabels=["Pred Normal", "Pred Failure"], yticklabels=["Actual Normal", "Actual Failure"])
plt.title("Held-Out Test Confusion Matrix (Threshold = 0.35)")
plt.show()


## 5. Inference Interface Test

Verify that the high-level inference interface seamlessly receives the 5 frontend sensor values and outputs correct risk classifications.

In [ ]:
from model_prediction import predict_failure_risk

# Nominal machine test
res_nominal = predict_failure_risk(air_temp=298.2, process_temp=308.7, rotational_speed=1500, torque=40.0, tool_wear=35)
print("Nominal Operation:", res_nominal["risk_level"], "| Prob:", res_nominal["failure_probability"])

# Overstrain machine test
res_overstrain = predict_failure_risk(air_temp=301.5, process_temp=310.8, rotational_speed=1300, torque=65.5, tool_wear=215)
print("Overstrain Condition:", res_overstrain["risk_level"], "| Prob:", res_overstrain["failure_probability"])
